# Final SALR and BCD runs: train, select, freeze (Nour)

Frozen configurations (chosen before any test evaluation):

| | value | source |
|---|---|---|
| `lambda_f, lambda_w` | 1e-3, 1e-3 | `regularization.ipynb` (shared by all four methods) |
| SALR | base_lr = 16, probe_step = 0.05, one-step probes, probe every update | `salr.ipynb` |
| BCD | max_inner = 1, inner_tol = 1e-4 | `bcd.ipynb` (tie with 1e-5 broken by less work) |
| budget | 1,000,000 triplet-gradients, checkpoint every 20,000 | common protocol |

For each seed 616/617/618: train, select the checkpoint with the best **ordinary-validation** NDCG@10,
save the JSON log to `results/logs/final/` and the selected parameters to `results/frozen/`.
`save_frozen` refuses to overwrite, so a frozen model can only be replaced deliberately.
**No test data is used in this notebook.**

In [1]:
import numpy as np, pandas as pd
from recommender.bcd import BCDConfig, train_bcd
from recommender.data import load_dataset
from recommender.evaluate import make_evaluator, select_checkpoint
from recommender.frozen import FROZEN_DIR, save_frozen
from recommender.runlog import LOG_DIR, save_log
from recommender.salr import SALRConfig, train_salr

ds = load_dataset()
evaluate = make_evaluator(ds)
SEEDS = (616, 617, 618)
LAMBDA_F = LAMBDA_W = 1e-3
TOTAL_WORK, CHECKPOINT_EVERY = 1_000_000, 20_000
FINAL = {
    "salr": (train_salr, SALRConfig(base_lr=16.0, probe_step=0.05, lambda_f=LAMBDA_F, lambda_w=LAMBDA_W)),
    "bcd": (train_bcd, BCDConfig(LAMBDA_F, LAMBDA_W, inner_tol=1e-4, max_inner=1)),
}
for name, (_, cfg) in FINAL.items():
    print(name, cfg)

salr SALRConfig(base_lr=16.0, probe_step=0.05, lambda_f=0.001, lambda_w=0.001, n_descent=1, n_ascent=1, probe_every=1, batch_size=256)
bcd BCDConfig(lambda_f=0.001, lambda_w=0.001, inner_tol=0.0001, max_inner=1, init_step=1.0, max_step=10000.0, armijo=0.0001, max_backtracks=40)


In [2]:
rows = []
for method, (train, cfg) in FINAL.items():
    for seed in SEEDS:
        log = train(ds, seed, cfg, TOTAL_WORK, CHECKPOINT_EVERY, evaluate)
        save_log(log, f"{method}_seed{seed}", LOG_DIR / "final", experiment="final")
        frozen = save_frozen(log)
        best = select_checkpoint(log)
        rows.append({
            "frozen_model": frozen, "status": log["status"],
            "val_ndcg": best["val_ndcg"], "val_recall": best["val_recall"],
            "val_loss": best["val_loss"], "train_loss": best["train_loss"],
            "selected_work": best["work"], "selected_updates": best["updates"],
            "train_time_to_selected_s": best["train_time"],
            "work_used": log["work"][-1], "train_time_total_s": log["checkpoints"][-1]["train_time"],
        })
final = pd.DataFrame(rows)
final.to_csv("results/final_salr_bcd_validation.csv", index=False)
final.round(4)

,frozen_model,status,val_ndcg,val_recall,val_loss,train_loss,selected_work,selected_updates,train_time_to_selected_s,work_used,train_time_total_s
0,salr_seed616,ok,0.1435,0.2645,0.5475,0.2351,900012,3642,8.5528,1000164,9.5471
1,salr_seed617,ok,0.1413,0.2314,0.5323,0.2378,860168,3481,8.4529,1000164,9.8327
2,salr_seed618,ok,0.1428,0.2397,0.5275,0.2410,480160,1943,4.4978,1000164,9.5754
3,bcd_seed616,converged,0.1376,0.2562,0.5651,0.3265,141336,26,0.8565,532728,3.5609
4,bcd_seed617,converged,0.1338,0.2645,0.5398,0.2529,184824,34,1.2053,565344,4.1700
5,bcd_seed618,converged,0.1316,0.2727,0.5385,0.3438,141336,26,0.9170,565344,4.2409


In [3]:
(final.assign(method=final.frozen_model.str.split("_").str[0])
      .groupby("method")
      .agg(val_ndcg=("val_ndcg", "mean"), val_ndcg_sd=("val_ndcg", "std"), val_recall=("val_recall", "mean"),
           selected_work=("selected_work", "mean"), time_to_selected_s=("train_time_to_selected_s", "mean"))
      .round(4))

,val_ndcg,val_ndcg_sd,val_recall,selected_work,time_to_selected_s
method,,,,,
bcd,0.1343,0.0031,0.2645,155832.0,0.9929
salr,0.1425,0.0011,0.2452,746780.0,7.1679


In [4]:
print("frozen models:", sorted(p.stem for p in FROZEN_DIR.glob("*.npz")))

frozen models: ['bcd_seed616', 'bcd_seed617', 'bcd_seed618', 'salr_seed616', 'salr_seed617', 'salr_seed618']
